# KNN, Naive Bayes & Decision Trees — Interview Assignment
**Topics:** K-Nearest Neighbors | Naive Bayes | Decision Trees  
**Format:** Conceptual questions → Coding challenges → Answers

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, load_breast_cancer, make_classification
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve
)
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
print('Libraries loaded successfully')

---
# PART 1: K-Nearest Neighbors (KNN)
---

## Section 1A: Conceptual Interview Questions

### Q1. What is the KNN algorithm? How does it make predictions?

**Answer:**  
KNN is a **lazy, non-parametric, instance-based** learning algorithm. It makes no assumptions about data distribution.

**Prediction steps:**
1. Store all training data points
2. For a new test point, compute distance to every training point
3. Select the K closest neighbors
4. **Classification:** majority vote among K neighbors
5. **Regression:** average of K neighbors' values

---

### Q2. Why is feature scaling critical in KNN?

**Answer:**  
KNN uses **distance metrics** (Euclidean, Manhattan, etc.). Features with larger scales dominate the distance calculation, making other features irrelevant.  
Example: Age (0–100) vs Income (0–1,000,000) — income will completely overshadow age without scaling.  
**Solution:** Always apply StandardScaler or MinMaxScaler before KNN.

---

### Q3. How do you choose the optimal value of K?

**Answer:**
- **Small K** → Low bias, High variance → Overfitting (model memorizes noise)
- **Large K** → High bias, Low variance → Underfitting (too smooth)
- **Best practice:** Use cross-validation / elbow method — plot error vs K and pick the K at the "elbow"
- Rule of thumb: Start with `K = sqrt(N)` where N = number of training samples
- Always prefer **odd K** for binary classification to avoid ties

---

### Q4. What are the distance metrics used in KNN?

**Answer:**
| Metric | Formula | Use Case |
|--------|---------|----------|
| Euclidean | `sqrt(Σ(xi-yi)²)` | Continuous, normally distributed features |
| Manhattan | `Σ|xi-yi|` | Robust to outliers, high-dimensional data |
| Minkowski | Generalization of both | Configurable via parameter `p` |
| Hamming | Proportion of disagreements | Categorical / binary data |
| Cosine | Angle between vectors | Text data, sparse high-dimensional data |

---

### Q5. What are the main disadvantages of KNN?

**Answer:**
- **Slow prediction:** O(N×D) per query — must scan all training data
- **High memory:** Stores entire training dataset
- **Curse of dimensionality:** Distances become meaningless in high dimensions
- **Sensitive to imbalanced data and outliers**
- **No model:** Cannot extract feature importance or business insights

## Section 1B: KNN Coding Questions

### Coding Q1: Implement KNN from scratch using only NumPy

In [ ]:
class KNNScratch:
    """KNN classifier built from scratch."""
    
    def __init__(self, k=3, distance='euclidean'):
        self.k = k
        self.distance = distance
    
    def fit(self, X, y):
        self.X_train = np.array(X)
        self.y_train = np.array(y)
        return self
    
    def _compute_distance(self, x1, x2):
        if self.distance == 'euclidean':
            return np.sqrt(np.sum((x1 - x2) ** 2))
        elif self.distance == 'manhattan':
            return np.sum(np.abs(x1 - x2))
        raise ValueError(f'Unknown distance: {self.distance}')
    
    def _predict_single(self, x):
        distances = [self._compute_distance(x, x_train) for x_train in self.X_train]
        k_indices = np.argsort(distances)[:self.k]
        k_labels = self.y_train[k_indices]
        # majority vote
        counts = np.bincount(k_labels.astype(int))
        return np.argmax(counts)
    
    def predict(self, X):
        return np.array([self._predict_single(x) for x in X])
    
    def score(self, X, y):
        return np.mean(self.predict(X) == y)


# --- Test against sklearn ---
iris = load_iris()
X, y = iris.data, iris.target

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Scratch implementation
knn_scratch = KNNScratch(k=5)
knn_scratch.fit(X_train, y_train)
scratch_acc = knn_scratch.score(X_test, y_test)

# sklearn implementation
knn_sklearn = KNeighborsClassifier(n_neighbors=5)
knn_sklearn.fit(X_train, y_train)
sklearn_acc = knn_sklearn.score(X_test, y_test)

print(f'Scratch KNN Accuracy : {scratch_acc:.4f}')
print(f'Sklearn KNN Accuracy : {sklearn_acc:.4f}')

### Coding Q2: Find optimal K using cross-validation (Elbow Method)

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

k_range = range(1, 31)
train_errors, val_errors = [], []

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

for k in k_range:
    pipe.set_params(knn__n_neighbors=k)
    pipe.fit(X_train, y_train)
    train_errors.append(1 - pipe.score(X_train, y_train))
    cv_score = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy').mean()
    val_errors.append(1 - cv_score)

optimal_k = k_range[np.argmin(val_errors)]

plt.figure(figsize=(12, 5))
plt.plot(k_range, train_errors, 'b-o', markersize=4, label='Train Error')
plt.plot(k_range, val_errors, 'r-o', markersize=4, label='CV Validation Error')
plt.axvline(x=optimal_k, color='green', linestyle='--', label=f'Optimal K={optimal_k}')
plt.xlabel('K (Number of Neighbors)')
plt.ylabel('Error Rate')
plt.title('KNN: Elbow Method for Optimal K')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Optimal K = {optimal_k}')
print(f'Min CV Error = {min(val_errors):.4f}')

### Coding Q3: Show the effect of feature scaling on KNN accuracy

In [ ]:
# Create dataset with vastly different scales
np.random.seed(42)
n = 300
age    = np.random.randint(18, 65, n).astype(float)
income = np.random.randint(20000, 120000, n).astype(float)
y_syn  = ((age > 40) & (income > 70000)).astype(int)

X_raw = np.column_stack([age, income])
X_tr_raw, X_te_raw, y_tr, y_te = train_test_split(X_raw, y_syn, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr_raw)
X_te_sc = scaler.transform(X_te_raw)

knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(X_tr_raw, y_tr)
acc_unscaled = knn.score(X_te_raw, y_te)

knn.fit(X_tr_sc, y_tr)
acc_scaled = knn.score(X_te_sc, y_te)

print('Feature Scaling Impact on KNN')
print('=' * 35)
print(f'Without Scaling : {acc_unscaled:.4f}')
print(f'With Scaling    : {acc_scaled:.4f}')
print(f'Improvement     : +{(acc_scaled - acc_unscaled):.4f}')

print('\nFeature ranges before scaling:')
print(f'  Age    : {age.min():.0f} - {age.max():.0f}')
print(f'  Income : {income.min():.0f} - {income.max():.0f}')

---
# PART 2: Naive Bayes
---

## Section 2A: Conceptual Interview Questions

### Q1. Explain the Naive Bayes algorithm. What makes it "naive"?

**Answer:**  
Naive Bayes is a **probabilistic classifier** based on Bayes' Theorem:

$$P(C|X) = \frac{P(X|C) \cdot P(C)}{P(X)}$$

Where:
- `P(C|X)` = Posterior — probability of class C given features X
- `P(X|C)` = Likelihood — probability of features given class C  
- `P(C)` = Prior — probability of class C
- `P(X)` = Evidence (constant, ignored during classification)

**"Naive" assumption:** All features are **conditionally independent** given the class label:  
`P(X|C) = P(x1|C) × P(x2|C) × ... × P(xn|C)`

This is rarely true in practice, but the algorithm works surprisingly well despite this.

---

### Q2. What are the different variants of Naive Bayes?

**Answer:**
| Variant | Distribution Assumed | Best For |
|---------|---------------------|----------|
| **GaussianNB** | Normal/Gaussian | Continuous features (height, weight) |
| **MultinomialNB** | Multinomial | Word counts, term frequencies |
| **BernoulliNB** | Bernoulli | Binary features (word present/absent) |
| **ComplementNB** | Complement of Multinomial | Imbalanced text data |

---

### Q3. What is Laplace Smoothing? Why is it needed?

**Answer:**  
If a feature value never appears with a class in training data, its likelihood `P(xi|C) = 0`, making the entire posterior probability **zero** regardless of other features — the **zero-frequency problem**.

**Laplace Smoothing** adds a small count `α` (usually 1) to all feature counts:

$$P(x_i|C) = \frac{\text{count}(x_i, C) + \alpha}{\text{count}(C) + \alpha \cdot |V|}$$

This ensures no probability is ever zero.

---

### Q4. When would you use Naive Bayes over other classifiers?

**Answer:**  
Use Naive Bayes when:
- **Text classification** (spam, sentiment, news categorization) — works great with bag-of-words
- **Very fast training required** — linear time complexity O(N×D)
- **Small training data** — low variance due to strong independence assumption
- **Real-time prediction** needed — trivial computation at inference
- **Interpretable probabilities** needed

---

### Q5. What are the limitations of Naive Bayes?

**Answer:**
- **Independence assumption** is almost always violated in real data
- **Bad probability estimates** — outputs probabilities that are overconfident (too close to 0 or 1)
- **Feature correlations** hurt performance significantly
- **Continuous features** must be binned or assumed Gaussian

## Section 2B: Naive Bayes Coding Questions

### Coding Q1: Implement Gaussian Naive Bayes from scratch

In [ ]:
class GaussianNBScratch:
    """Gaussian Naive Bayes from scratch."""
    
    def fit(self, X, y):
        X, y = np.array(X), np.array(y)
        self.classes = np.unique(y)
        self.priors = {}
        self.means = {}
        self.stds = {}
        
        for c in self.classes:
            X_c = X[y == c]
            self.priors[c] = len(X_c) / len(X)
            self.means[c] = X_c.mean(axis=0)
            self.stds[c] = X_c.std(axis=0) + 1e-9  # avoid division by zero
        return self
    
    def _gaussian_log_likelihood(self, x, mean, std):
        # log of Gaussian PDF — avoids numerical underflow
        return -0.5 * np.log(2 * np.pi * std**2) - ((x - mean)**2) / (2 * std**2)
    
    def _predict_single(self, x):
        log_posteriors = {}
        for c in self.classes:
            log_prior = np.log(self.priors[c])
            log_likelihood = self._gaussian_log_likelihood(x, self.means[c], self.stds[c]).sum()
            log_posteriors[c] = log_prior + log_likelihood
        return max(log_posteriors, key=log_posteriors.get)
    
    def predict(self, X):
        return np.array([self._predict_single(x) for x in X])
    
    def score(self, X, y):
        return np.mean(self.predict(X) == y)


# --- Compare with sklearn ---
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

nb_scratch = GaussianNBScratch().fit(X_train, y_train)
nb_sklearn = GaussianNB().fit(X_train, y_train)

print('Gaussian Naive Bayes — Scratch vs Sklearn')
print('=' * 45)
print(f'Scratch Accuracy : {nb_scratch.score(X_test, y_test):.4f}')
print(f'Sklearn Accuracy : {nb_sklearn.score(X_test, y_test):.4f}')

print('\nClass Priors (Scratch):')
for c, p in nb_scratch.priors.items():
    print(f'  Class {c}: {p:.3f}')

### Coding Q2: Spam classifier using Multinomial Naive Bayes

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Synthetic email dataset
emails = [
    # spam
    'win free money now click here',
    'congratulations you won a prize claim now',
    'free offer limited time buy now',
    'earn money from home fast cash',
    'click to claim your free prize today',
    'urgent winner selected claim prize immediately',
    'make money fast work from home',
    'special offer buy now save money',
    # ham
    'meeting scheduled for tomorrow morning',
    'please review the attached project report',
    'can we reschedule our call to afternoon',
    'team lunch this friday at noon',
    'your code review has been approved',
    'please submit the expense report by friday',
    'quarterly review meeting is next monday',
    'thank you for your help on this project',
]
labels = [1]*8 + [0]*8  # 1=spam, 0=ham

# Vectorize
vectorizer = TfidfVectorizer()
X_vec = vectorizer.fit_transform(emails)
y = np.array(labels)

X_train, X_test, y_train, y_test = train_test_split(X_vec, y, test_size=0.25, random_state=42)

# Train MultinomialNB (needs non-negative features — use CountVectorizer or clip TF-IDF)
count_vec = CountVectorizer()
X_count = count_vec.fit_transform(emails)
X_tr_c, X_te_c, y_tr, y_te = train_test_split(X_count, y, test_size=0.25, random_state=42)

mnb = MultinomialNB(alpha=1.0)  # alpha=1 → Laplace smoothing
mnb.fit(X_tr_c, y_tr)

print('Spam Classifier — Multinomial Naive Bayes')
print('=' * 45)
print(f'Accuracy: {mnb.score(X_te_c, y_te):.4f}')
print()

# Test on new emails
new_emails = [
    'win free money click here now',     # should be spam
    'meeting is scheduled for monday',   # should be ham
]
X_new = count_vec.transform(new_emails)
predictions = mnb.predict(X_new)
probs = mnb.predict_proba(X_new)

for email, pred, prob in zip(new_emails, predictions, probs):
    label = 'SPAM' if pred == 1 else 'HAM'
    print(f'Email : "{email}"')
    print(f'Prediction: {label} | P(ham)={prob[0]:.3f} | P(spam)={prob[1]:.3f}')
    print()

### Coding Q3: Compare GaussianNB variants and visualize decision boundary

In [ ]:
from sklearn.datasets import make_moons, make_blobs

# 2D dataset for visualization
X_blobs, y_blobs = make_blobs(n_samples=300, centers=3, cluster_std=1.0, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_blobs, y_blobs, test_size=0.2, random_state=42)

models = {
    'GaussianNB': GaussianNB(),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, model) in zip(axes, models.items()):
    model.fit(X_tr, y_tr)
    acc = model.score(X_te, y_te)
    
    x_min, x_max = X_blobs[:, 0].min() - 1, X_blobs[:, 0].max() + 1
    y_min, y_max = X_blobs[:, 1].min() - 1, X_blobs[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                          np.linspace(y_min, y_max, 200))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
    scatter = ax.scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_blobs,
                         cmap='RdYlBu', edgecolors='k', s=40)
    ax.set_title(f'{name}\nAccuracy: {acc:.4f}', fontsize=13)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

plt.suptitle('Decision Boundary: GaussianNB vs KNN', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
# PART 3: Decision Trees
---

## Section 3A: Conceptual Interview Questions

### Q1. How does a Decision Tree decide which feature to split on?

**Answer:**  
Decision trees choose splits that maximize **information gain** (or minimize impurity).

Two main impurity measures:

**Gini Impurity (used by CART / sklearn default):**
$$Gini = 1 - \sum_{i=1}^{C} p_i^2$$

**Entropy (used by ID3, C4.5):**
$$H = -\sum_{i=1}^{C} p_i \log_2(p_i)$$

**Information Gain:**
$$IG = H(parent) - \sum_k \frac{n_k}{n} H(child_k)$$

The split with the highest IG is chosen at each node.

---

### Q2. What is overfitting in Decision Trees and how do you prevent it?

**Answer:**  
An unpruned tree can perfectly memorize training data by creating one leaf per sample — **zero training error but poor generalization**.

**Prevention techniques:**
| Technique | Parameter | Effect |
|-----------|-----------|--------|
| Max Depth | `max_depth` | Limits tree height |
| Min Samples Split | `min_samples_split` | Prevents splitting tiny nodes |
| Min Samples Leaf | `min_samples_leaf` | Ensures leaves have enough data |
| Max Features | `max_features` | Considers only a subset of features per split |
| Pruning (Post) | `ccp_alpha` | Cost-complexity pruning |

---

### Q3. What is the difference between Gini and Entropy?

**Answer:**
- Both measure **impurity** — how mixed the classes are at a node
- Gini range: `[0, 0.5]` for binary; Entropy range: `[0, 1]`
- Gini is **computationally faster** (no logarithm)
- In practice they produce **nearly identical trees**
- Entropy is slightly better at finding balanced splits
- Gini tends to isolate the most frequent class into its own branch

---

### Q4. Can Decision Trees handle missing values and categorical features natively?

**Answer:**
- **sklearn's implementation:** Cannot handle missing values or raw categorical features natively — requires preprocessing
- **Original CART:** Has surrogate splits for missing values
- **C4.5/C5.0:** Can handle missing values and categorical features
- **Best practice with sklearn:** Use `SimpleImputer` for missing values and `OrdinalEncoder` / `OneHotEncoder` for categoricals

---

### Q5. What is the time complexity for training and predicting with a Decision Tree?

**Answer:**
- **Training:** O(N × D × log N) where N = samples, D = features
  - For each node: sort D features × check N split points
  - Tree has O(log N) depth on average
- **Prediction:** O(log N) per sample — just traverse the tree
- **Memory:** O(N) to store the tree structure

## Section 3B: Decision Trees Coding Questions

### Coding Q1: Implement Gini Impurity and Information Gain from scratch

In [ ]:
def gini_impurity(y):
    """Gini = 1 - sum(p_i^2)"""
    if len(y) == 0:
        return 0
    classes, counts = np.unique(y, return_counts=True)
    probs = counts / len(y)
    return 1 - np.sum(probs ** 2)


def entropy(y):
    """H = -sum(p_i * log2(p_i))"""
    if len(y) == 0:
        return 0
    classes, counts = np.unique(y, return_counts=True)
    probs = counts / len(y)
    # clip to avoid log(0)
    return -np.sum(probs * np.log2(np.clip(probs, 1e-10, 1)))


def information_gain(y_parent, y_left, y_right, criterion='entropy'):
    """IG = H(parent) - weighted_avg(H(children))"""
    fn = entropy if criterion == 'entropy' else gini_impurity
    n = len(y_parent)
    n_l, n_r = len(y_left), len(y_right)
    parent_impurity = fn(y_parent)
    weighted_child = (n_l / n) * fn(y_left) + (n_r / n) * fn(y_right)
    return parent_impurity - weighted_child


# --- Demonstration ---
# Perfect split: [0,0,0] and [1,1,1]
y_parent = np.array([0, 0, 0, 1, 1, 1])
y_left   = np.array([0, 0, 0])
y_right  = np.array([1, 1, 1])

# Impure split: [0,1,0] and [1,0,1]
y_left_bad  = np.array([0, 1, 0])
y_right_bad = np.array([1, 0, 1])

print('Impurity Measures Demonstration')
print('=' * 45)
print(f'Pure node [0,0,0]       → Gini: {gini_impurity(y_left):.4f} | Entropy: {entropy(y_left):.4f}')
print(f'Mixed node [0,1,0,1,0,1] → Gini: {gini_impurity(y_parent):.4f} | Entropy: {entropy(y_parent):.4f}')
print()
print(f'Perfect split IG (entropy) : {information_gain(y_parent, y_left, y_right):.4f}')
print(f'Bad split IG (entropy)     : {information_gain(y_parent, y_left_bad, y_right_bad):.4f}')
print()
print('Key insight: Perfect split → IG = entropy of parent (maximum possible)')
print('             Bad split     → IG ≈ 0 (no information gained)')

### Coding Q2: Train, visualize and interpret a Decision Tree

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
feature_names = iris.feature_names
class_names = iris.target_names

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

dt = DecisionTreeClassifier(max_depth=3, criterion='gini', random_state=42)
dt.fit(X_train, y_train)

print('Decision Tree Performance')
print('=' * 35)
print(f'Train Accuracy: {dt.score(X_train, y_train):.4f}')
print(f'Test Accuracy : {dt.score(X_test, y_test):.4f}')
print()

# Text representation
print('Tree Structure (text):')
print(export_text(dt, feature_names=feature_names))

# Visual tree
fig, ax = plt.subplots(figsize=(16, 8))
plot_tree(dt, feature_names=feature_names, class_names=class_names,
          filled=True, rounded=True, fontsize=10, ax=ax)
plt.title('Decision Tree Visualization (max_depth=3, Iris Dataset)', fontsize=14)
plt.tight_layout()
plt.show()

# Feature importance
print('\nFeature Importances:')
for name, imp in sorted(zip(feature_names, dt.feature_importances_),
                         key=lambda x: -x[1]):
    bar = '█' * int(imp * 40)
    print(f'  {name:30s}: {imp:.4f}  {bar}')

### Coding Q3: Hyperparameter tuning with GridSearchCV and overfitting analysis

In [ ]:
bc = load_breast_cancer()
X, y = bc.data, bc.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- Overfitting analysis: train/test accuracy vs max_depth ---
depths = range(1, 20)
train_accs, test_accs = [], []

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train, y_train)
    train_accs.append(dt.score(X_train, y_train))
    test_accs.append(dt.score(X_test, y_test))

best_depth = depths[np.argmax(test_accs)]

plt.figure(figsize=(12, 5))
plt.plot(depths, train_accs, 'b-o', markersize=5, label='Train Accuracy')
plt.plot(depths, test_accs, 'r-o', markersize=5, label='Test Accuracy')
plt.axvline(x=best_depth, color='green', linestyle='--', label=f'Best Depth={best_depth}')
plt.fill_between(depths, train_accs, test_accs, alpha=0.1, color='red', label='Overfitting Gap')
plt.xlabel('Max Depth')
plt.ylabel('Accuracy')
plt.title('Decision Tree: Overfitting Analysis (Breast Cancer Dataset)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Best max_depth: {best_depth}')
print(f'Best Test Accuracy: {max(test_accs):.4f}')

# --- GridSearchCV for best hyperparameters ---
param_grid = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}

grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid, cv=5, scoring='accuracy', n_jobs=-1
)
grid_search.fit(X_train, y_train)

print(f'\nGridSearchCV Best Params: {grid_search.best_params_}')
print(f'GridSearchCV CV Score  : {grid_search.best_score_:.4f}')
print(f'Test Accuracy          : {grid_search.best_estimator_.score(X_test, y_test):.4f}')

### Coding Q4: Cost-Complexity Pruning (Post-Pruning)

In [ ]:
bc = load_breast_cancer()
X, y = bc.data, bc.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Get the pruning path — all valid ccp_alpha values
dt_full = DecisionTreeClassifier(random_state=42)
path = dt_full.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas[:-1]  # remove last (trivial single-node tree)

train_scores, test_scores, n_nodes = [], [], []

for alpha in ccp_alphas:
    dt = DecisionTreeClassifier(ccp_alpha=alpha, random_state=42)
    dt.fit(X_train, y_train)
    train_scores.append(dt.score(X_train, y_train))
    test_scores.append(dt.score(X_test, y_test))
    n_nodes.append(dt.tree_.node_count)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(ccp_alphas, train_scores, 'b-', label='Train')
ax1.plot(ccp_alphas, test_scores, 'r-', label='Test')
ax1.set_xlabel('ccp_alpha')
ax1.set_ylabel('Accuracy')
ax1.set_title('Accuracy vs ccp_alpha')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(ccp_alphas, n_nodes, 'g-o', markersize=3)
ax2.set_xlabel('ccp_alpha')
ax2.set_ylabel('Number of Nodes')
ax2.set_title('Tree Size vs ccp_alpha')
ax2.grid(True, alpha=0.3)

plt.suptitle('Cost-Complexity Pruning Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

best_alpha = ccp_alphas[np.argmax(test_scores)]
print(f'Best ccp_alpha : {best_alpha:.6f}')
print(f'Best Test Acc  : {max(test_scores):.4f}')
print(f'Tree nodes at best alpha: {n_nodes[np.argmax(test_scores)]}')

---
# PART 4: Head-to-Head Comparison
---

## Section 4A: Conceptual Comparison Questions

### Q1. When would you choose each algorithm?

| Scenario | Best Choice | Reason |
|----------|------------|--------|
| Text classification (spam/sentiment) | Naive Bayes | Fast, works well with word features |
| Small dataset, non-parametric needed | KNN | No training phase, flexible boundary |
| Need interpretability / feature importance | Decision Tree | Rules are human-readable |
| High-dimensional data | Naive Bayes | Handles curse of dimensionality better |
| Real-time serving at scale | Decision Tree | O(log N) prediction, no feature scaling |
| Noisy data with outliers | Decision Tree | Gini/entropy are robust to outliers |

---

### Q2. How does each algorithm handle the bias-variance tradeoff?

| Algorithm | Bias | Variance | Notes |
|-----------|------|----------|-------|
| KNN (small k) | Low | High | Overfits to local noise |
| KNN (large k) | High | Low | Too smooth, underfits |
| Naive Bayes | High | Low | Strong independence assumption |
| Decision Tree (deep) | Low | High | Memorizes training data |
| Decision Tree (pruned) | Medium | Medium | Balanced with regularization |

## Section 4B: All Three Models on Same Dataset

In [ ]:
# Benchmark all three on Breast Cancer dataset
bc = load_breast_cancer()
X, y = bc.data, bc.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)

models = {
    'KNN (k=5, scaled)': KNeighborsClassifier(n_neighbors=5),
    'Gaussian Naive Bayes': GaussianNB(),
    'Decision Tree (depth=5)': DecisionTreeClassifier(max_depth=5, random_state=42),
}

results = {}
for name, model in models.items():
    if 'KNN' in name:
        model.fit(X_tr_sc, y_train)
        y_pred = model.predict(X_te_sc)
        y_prob = model.predict_proba(X_te_sc)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]
    
    results[name] = {
        'accuracy': accuracy_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_prob),
        'cv_score': cross_val_score(model,
                                    X_tr_sc if 'KNN' in name else X_train,
                                    y_train, cv=5).mean(),
        'y_prob': y_prob
    }

# Print comparison table
print('Model Benchmark — Breast Cancer Dataset')
print('=' * 65)
print(f'{"Model":<30} {"Accuracy":>10} {"ROC-AUC":>10} {"5-Fold CV":>10}')
print('-' * 65)
for name, res in results.items():
    print(f'{name:<30} {res["accuracy"]:>10.4f} {res["roc_auc"]:>10.4f} {res["cv_score"]:>10.4f}')

# ROC Curves
plt.figure(figsize=(9, 6))
colors = ['blue', 'orange', 'green']
for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    plt.plot(fpr, tpr, color=color,
             label=f'{name} (AUC={res["roc_auc"]:.4f})', linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison: KNN vs Naive Bayes vs Decision Tree')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrices side by side
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (name, res) in zip(axes, results.items()):
    if 'KNN' in name:
        model = models[name]
        y_pred = model.predict(X_te_sc)
    else:
        model = models[name]
        y_pred = model.predict(X_test)
    
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Malignant', 'Benign'],
                yticklabels=['Malignant', 'Benign'])
    ax.set_title(f'{name}\nAcc: {res["accuracy"]:.4f}', fontsize=10)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices — Breast Cancer Classification', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
# PART 5: Advanced Interview Questions
---

### Q1. How do Decision Trees handle multiclass classification?

**Answer:**  
Decision Trees naturally handle multiclass — at each leaf, the majority class among training samples becomes the prediction. No modification needed (unlike SVM which requires OvR/OvO strategies).

---

### Q2. Can KNN be used for regression? How?

**Answer:**  
Yes — `sklearn.neighbors.KNeighborsRegressor`. Instead of majority vote, it predicts the **average** (or weighted average) of the K nearest neighbors' target values. Useful for local interpolation but suffers the same scaling/dimensionality issues.

---

### Q3. Why does Naive Bayes work well for text despite its naive assumption?

**Answer:**  
Even though words ARE correlated ("machine" and "learning" often co-occur), NB still works because:
1. For **classification** we only need to identify which class has the higher posterior — not the exact probability values
2. The **ranking** of classes is often correct even when exact probabilities are wrong
3. Text features are **sparse** — most words don't appear in any given document, so correlations are limited

---

### Q4. What is the relationship between Decision Trees and Random Forests?

**Answer:**  
Random Forest is an **ensemble of decision trees** that reduces overfitting through:
1. **Bagging:** Each tree trains on a bootstrap sample of the data
2. **Feature randomness:** Each split considers only a random subset of features (`max_features = sqrt(D)`)
3. **Aggregation:** Final prediction is majority vote (classification) or mean (regression)

This dramatically reduces variance while maintaining low bias — fixing the main weakness of individual trees.

---

### Q5. How would you handle class imbalance in each algorithm?

| Algorithm | Strategy |
|-----------|----------|
| KNN | Use `weights='distance'`; oversample minority class (SMOTE) |
| Naive Bayes | Adjust class priors manually; use class_prior parameter |
| Decision Tree | Set `class_weight='balanced'`; adjust decision threshold |

### Bonus Coding: Class Imbalance Handling

In [ ]:
from sklearn.datasets import make_classification

# Create imbalanced dataset (90% class 0, 10% class 1)
X_imb, y_imb = make_classification(
    n_samples=1000, n_features=10, weights=[0.9, 0.1],
    random_state=42
)

X_tr, X_te, y_tr, y_te = train_test_split(X_imb, y_imb, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr)
X_te_sc = scaler.transform(X_te)

print('Class distribution in training set:')
unique, counts = np.unique(y_tr, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  Class {u}: {c} ({100*c/len(y_tr):.1f}%)')
print()

models_imb = {
    'KNN (uniform)': KNeighborsClassifier(n_neighbors=5, weights='uniform'),
    'KNN (distance)': KNeighborsClassifier(n_neighbors=5, weights='distance'),
    'GaussianNB (default)': GaussianNB(),
    'GaussianNB (priors)': GaussianNB(priors=[0.5, 0.5]),  # manual prior balancing
    'DT (default)': DecisionTreeClassifier(max_depth=5, random_state=42),
    'DT (balanced)': DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42),
}

print(f'{'Model':<30} {'Accuracy':>10} {'ROC-AUC':>10}')
print('-' * 55)
for name, model in models_imb.items():
    if 'KNN' in name or 'NB' in name:
        model.fit(X_tr_sc, y_tr)
        y_pred = model.predict(X_te_sc)
        y_prob = model.predict_proba(X_te_sc)[:, 1]
    else:
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)
        y_prob = model.predict_proba(X_te)[:, 1]
    
    acc = accuracy_score(y_te, y_pred)
    auc = roc_auc_score(y_te, y_prob)
    print(f'{name:<30} {acc:>10.4f} {auc:>10.4f}')

---
# Quick Reference Summary

| Property | KNN | Naive Bayes | Decision Tree |
|----------|-----|-------------|---------------|
| **Type** | Instance-based | Probabilistic | Rule-based |
| **Training time** | O(1) | O(N×D) | O(N×D×logN) |
| **Prediction time** | O(N×D) | O(D) | O(log N) |
| **Feature scaling** | Required | Not needed | Not needed |
| **Missing values** | Needs imputation | Needs imputation | Needs imputation (sklearn) |
| **Interpretable** | No | Partial | Yes |
| **Handles non-linear** | Yes | No | Yes |
| **Sensitive to outliers** | Yes | No | Low |
| **Key hyperparameter** | k | alpha (smoothing) | max_depth |
| **Best use case** | Recommendation, anomaly | Text, spam filter | Business rules, tabular |